# SKGB MINE Benchmark

Evaluate the SKGB KG construction pipeline against the **MINE** benchmark (NeurIPS 2025).

- **Dataset:** 100 articles, 15 ground-truth facts each (HuggingFace: `josancamon/kg-gen-MINE-evaluation-dataset`)
- **Protocol:** Build KG per article -> embed nodes -> retrieve context for each fact -> LLM judges if fact is inferable
- **Score:** `count(inferable facts) / 15` per article, averaged across all articles
- **LLM:** minimax-m2.5:cloud via Ollama official API

### Fixes applied
- Explicit `num_ctx` passed to all LangChain `ChatOllama` and direct Ollama API calls
- Reduced chunk sizes to stay within context window safely
- Serial worker mode (`MAX_WORKERS=1`) to avoid cloud proxy rate-limiting
- Model changed to `minimax-m2.5:cloud` (198K context window via Ollama official API)
- Fixed monkey-patching to avoid recursion errors

In [ ]:
# Cell 1: Install dependencies
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    if not os.path.exists("/content/DynamicKGConstruction"):
        subprocess.check_call(
            ["git", "clone", "https://github.com/edwinidrus/DynamicKGConstruction.git"],
            cwd="/content",
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    print("Repo ready at /content/DynamicKGConstruction")


def run_pip(args):
    subprocess.check_call([sys.executable, "-m", "pip", *args])


print("Resetting scientific Python stack to avoid NumPy/SciPy binary mismatch...")
run_pip(["uninstall", "-y", "numpy", "scipy", "pandas", "scikit-learn"])
run_pip(["cache", "purge"])

core_stack = [
    "numpy<2.0",
    "pandas>=2.0,<2.3",
    "scipy>=1.11,<1.14",
    "scikit-learn>=1.4,<1.6",
]
run_pip(["install", "-q", "--no-cache-dir", "--force-reinstall", *core_stack])

notebook_deps = [
    "itext2kg==1.0.0",
    "langchain>=0.3.26,<0.4.0",
    "langchain-core>=0.3.69,<0.4.0",
    "langchain-ollama>=0.1.0,<1.0.0",
    "networkx",
    "datasets",
    "sentence-transformers",
    "tqdm",
    "nest_asyncio",
    "python-dotenv",
    "ollama",
    "matplotlib",
    "seaborn",
]
run_pip(["install", "-q", "--no-cache-dir", *notebook_deps])

print("Dependency bootstrap complete.")
print("If this cell changed package versions in an active kernel, restart the kernel before running Cell 2.")

Repo cloned to /content/DynamicKGConstruction
Dependencies installed.


## Environment note

Cell 1 now rebuilds the scientific Python stack in a deterministic order so `datasets` can import `scipy` without hitting the NumPy ABI mismatch seen previously.

If Cell 1 reinstalls packages, restart the kernel once before continuing with Cell 2.

In [2]:
# Cell 2: Imports & path setup
import sys, os, json, datetime, tempfile, ast, importlib, types
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    SKGB_ROOT = Path("/content/DynamicKGConstruction")
else:
    # Local: notebook lives in DynamicKGConstruction/notebooks/
    SKGB_ROOT = Path("..").resolve()
    if not (SKGB_ROOT / "skgb").is_dir():
        # Windows fallback
        SKGB_ROOT = Path(r"C:\Users\EdwinH\OneDrive\Dokumente\PhD\repo\DynamicKG\DynamicKGConstruction")

if str(SKGB_ROOT) not in sys.path:
    sys.path.insert(0, str(SKGB_ROOT))

# Register "skgb" as an empty namespace so sub-module imports skip __init__.py
# (avoids importing docling which is heavy and not needed for benchmarking)
if "skgb" not in sys.modules:
    _pkg = types.ModuleType("skgb")
    _pkg.__path__ = [str(SKGB_ROOT / "skgb")]
    _pkg.__package__ = "skgb"
    sys.modules["skgb"] = _pkg

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

print(f"SKGB root: {SKGB_ROOT}")
print(f"IN_COLAB:  {IN_COLAB}")
print(f"Python:    {sys.executable}")

SKGB root: /content/DynamicKGConstruction
IN_COLAB:  True
Python:    /usr/bin/python3


In [ ]:
# Cell 3: Cloud Ollama authentication
#
# Three-tier API key resolution:
#   1. .env file (local)
#   2. Colab Secrets (google.colab.userdata)
#   3. Interactive prompt (getpass)

import getpass
import os

import ollama

OLLAMA_BASE_URL = "https://api.ollama.com"
OLLAMA_API_KEY = ""
LLM_MODEL = "minimax-m2.5:cloud"

try:
    from dotenv import load_dotenv

    env_path = SKGB_ROOT / "notebooks" / ".env"
    if env_path.exists():
        load_dotenv(dotenv_path=env_path, override=True)
        OLLAMA_API_KEY = os.environ.get("ollamaApiKey", "") or os.environ.get("OLLAMA_API_KEY", "")
        if OLLAMA_API_KEY:
            print(f"API key loaded from .env ({env_path.name})")
except ImportError:
    pass

if not OLLAMA_API_KEY and IN_COLAB:
    try:
        from google.colab import userdata

        OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY")
        if OLLAMA_API_KEY:
            print("API key loaded from Colab Secrets")
    except Exception:
        pass

if not OLLAMA_API_KEY:
    OLLAMA_API_KEY = getpass.getpass("Enter Ollama API key: ")

assert OLLAMA_API_KEY, "No API key found!"
print(f"Ollama endpoint: {OLLAMA_BASE_URL}")
print(f"Model:           {LLM_MODEL}")
print(f"API key loaded:  {OLLAMA_API_KEY[:8]}...")

_original_async_client = ollama.AsyncClient
_original_sync_client = ollama.Client
_original_chat = ollama.chat


class PatchedAsyncClient(_original_async_client):
    def __init__(self, *args, **kwargs):
        headers = dict(kwargs.get("headers") or {})
        headers["Authorization"] = f"Bearer {OLLAMA_API_KEY}"
        kwargs["headers"] = headers
        super().__init__(*args, **kwargs)


class PatchedSyncClient(_original_sync_client):
    def __init__(self, *args, **kwargs):
        headers = dict(kwargs.get("headers") or {})
        headers["Authorization"] = f"Bearer {OLLAMA_API_KEY}"
        kwargs["headers"] = headers
        super().__init__(*args, **kwargs)


ollama.AsyncClient = PatchedAsyncClient
ollama.Client = PatchedSyncClient


def _patched_chat(model, messages, **kwargs):
    headers = dict(kwargs.get("headers") or {})
    headers["Authorization"] = f"Bearer {OLLAMA_API_KEY}"
    kwargs["headers"] = headers
    return _original_chat(model, messages, **kwargs)


ollama.chat = _patched_chat

try:
    client = ollama.Client(host=OLLAMA_BASE_URL)
    response = client.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "Reply with the word ready."}],
        options={"num_ctx": 1024, "temperature": 0.0},
    )
    print(f"Connected. Probe response: {response['message']['content'][:60]}...")
except Exception as e:
    print(f"WARNING: Ollama connection failed: {e}")
    print("Continue only after confirming the endpoint and API key are valid.")

Enter Ollama API key: ··········
Ollama endpoint: https://api.ollama.com
API key loaded:  773693d0...
Connected! Model responded: Hello! How can I help you today?...


In [ ]:
# Cell 4: Configuration

EMBEDDINGS_MODEL = "nomic-embed-text"
TEMPERATURE = 0.0
ENT_THRESHOLD = 0.8
REL_THRESHOLD = 0.7
MAX_WORKERS = 1
MIN_CHUNK_WORDS = 100
MAX_CHUNK_WORDS = 400
OVERLAP_WORDS = 50
PRESERVE_METADATA = False
NUM_CTX = 16384

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
TOP_K = 5
HOP_DEPTH = 2

MODEL_RUN_TAG = LLM_MODEL.replace(":", "-").replace("/", "-")
RESULTS_DIR = Path("benchmark_results") / "mine" / MODEL_RUN_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"LLM: {LLM_MODEL} @ {OLLAMA_BASE_URL}")
print(f"Context window (num_ctx): {NUM_CTX}")
print(f"Chunk words: {MIN_CHUNK_WORDS}-{MAX_CHUNK_WORDS} (overlap {OVERLAP_WORDS})")
print(f"Preserve metadata chunk: {PRESERVE_METADATA}")
print(f"Max workers: {MAX_WORKERS}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

LLM: qwen3.5:397b-cloud @ https://api.ollama.com
Context window (num_ctx): 16384
Chunk words: 100-400 (overlap 50)
Max workers: 1
Results dir: /content/benchmark_results/mine


In [ ]:
# Cell 5: Validate environment and load MINE dataset

import importlib

for package_name in ["numpy", "pandas", "scipy", "sklearn", "datasets"]:
    module = importlib.import_module(package_name)
    version = getattr(module, "__version__", "unknown")
    print(f"{package_name:<10} {version}")

from datasets import load_dataset

ds = load_dataset("josancamon/kg-gen-MINE-evaluation-dataset", split="train")
print(f"\nLoaded {len(ds)} articles")
print(f"Columns: {ds.column_names}")

row = ds[0]
print(f"\nTopic: {row['essay_topic']}")
print(f"Essay length: {len(row['essay_content'].split())} words")

facts_raw = row["generated_queries"]
if isinstance(facts_raw, str):
    facts = ast.literal_eval(facts_raw)
else:
    facts = list(facts_raw)
print(f"Facts ({len(facts)}): {facts[:3]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

In [ ]:
# Cell 6: SKGB imports + helper

import importlib.util
import types

for subpkg in ("adapters", "export", "utils"):
    mod_name = f"skgb.{subpkg}"
    if mod_name not in sys.modules:
        mod = types.ModuleType(mod_name)
        mod.__path__ = [str(SKGB_ROOT / "skgb" / subpkg)]
        mod.__package__ = mod_name
        sys.modules[mod_name] = mod

if "skgb.models" not in sys.modules:
    _spec = importlib.util.spec_from_file_location(
        "skgb.models", str(SKGB_ROOT / "skgb" / "models.py")
    )
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules["skgb.models"] = _mod
    _spec.loader.exec_module(_mod)

from skgb.adapters.chunking_adapter import chunk_markdown_files
from skgb.adapters.itext2kg_adapter import build_kg_from_atomic_facts
from skgb.export.file_export import kg_to_dict

try:
    import itext2kg.utils.llm as itext2kg_llm

    print("itext2kg provider module detected:", itext2kg_llm.__file__)
except Exception as exc:
    print(f"WARNING: itext2kg provider module unavailable: {exc}")

try:
    from langchain_ollama import ChatOllama as _OrigChatOllama

    _orig_chat_init = _OrigChatOllama.__init__

    def _patched_chat_init(self, *args, **kwargs):
        if "num_ctx" not in kwargs:
            kwargs["num_ctx"] = NUM_CTX
        _orig_chat_init(self, *args, **kwargs)

    _OrigChatOllama.__init__ = _patched_chat_init
    print(f"Patched ChatOllama to inject num_ctx={NUM_CTX}")
except ImportError:
    print("WARNING: Could not patch ChatOllama — langchain_ollama not found")

try:
    from langchain_ollama import OllamaEmbeddings as _OrigOllamaEmbed

    _orig_embed_init = _OrigOllamaEmbed.__init__

    def _patched_embed_init(self, *args, **kwargs):
        if "num_ctx" not in kwargs:
            kwargs["num_ctx"] = 8192
        _orig_embed_init(self, *args, **kwargs)

    _OrigOllamaEmbed.__init__ = _patched_embed_init
    print("Patched OllamaEmbeddings to inject num_ctx=8192")
except (ImportError, Exception) as e:
    print(f"Note: OllamaEmbeddings patch skipped ({e})")


def inspect_chunks(text: str) -> list[dict]:
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".md", delete=False, encoding="utf-8"
    ) as f:
        f.write(text)
        tmp_path = Path(f.name)

    try:
        chunks = chunk_markdown_files(
            md_paths=[tmp_path],
            min_chunk_words=MIN_CHUNK_WORDS,
            max_chunk_words=MAX_CHUNK_WORDS,
            overlap_words=OVERLAP_WORDS,
            preserve_metadata=PRESERVE_METADATA,
        )
        return chunks
    finally:
        tmp_path.unlink(missing_ok=True)


def run_skgb_on_text(text: str, *, debug: bool = False) -> dict:
    """Run the benchmark path on plain text while keeping chunk diagnostics visible."""
    chunks = inspect_chunks(text)

    t_obs = datetime.datetime.now().strftime("%Y-%m-%d")
    atomic_facts_dict = {t_obs: []}
    for ch in chunks:
        section_title = ch.get("section_title") or ""
        content = ch.get("content") or ""
        atomic_facts_dict[t_obs].append(f"[{section_title}] {content}".strip())

    avg_words = sum(len(c.split()) for c in atomic_facts_dict[t_obs]) // max(len(atomic_facts_dict[t_obs]), 1)
    print(f"  Chunks: {len(atomic_facts_dict[t_obs])} (avg {avg_words} words)")
    if debug:
        for index, chunk in enumerate(chunks, start=1):
            preview = (chunk.get("content") or "").replace("\n", " ")[:160]
            print(f"    Chunk {index}: {chunk.get('section_title', '')} :: {preview}")

    kg = build_kg_from_atomic_facts(
        atomic_facts_dict=atomic_facts_dict,
        ollama_base_url=OLLAMA_BASE_URL,
        llm_model=LLM_MODEL,
        embeddings_model=EMBEDDINGS_MODEL,
        temperature=TEMPERATURE,
        ent_threshold=ENT_THRESHOLD,
        rel_threshold=REL_THRESHOLD,
        max_workers=MAX_WORKERS,
    )

    return kg_to_dict(kg)

print("SKGB helpers loaded.")

In [ ]:
# Cell 7: Smoke test — run SKGB on the first article without cache

print(f"Running uncached smoke test on article 0: {ds[0]['essay_topic']}")
print(f"Essay word count: {len(ds[0]['essay_content'].split())}")
print(f"Model: {LLM_MODEL}, num_ctx: {NUM_CTX}")
print(f"Results dir for later cached runs: {RESULTS_DIR}")
print()

try:
    kg_dict = run_skgb_on_text(ds[0]["essay_content"], debug=True)
    print(f"\nNodes: {len(kg_dict['nodes'])}, Edges: {len(kg_dict['edges'])}")

    for edge in kg_dict["edges"][:5]:
        print(f"  {edge['source']} --[{edge['relation']}]--> {edge['target']}")
except Exception as e:
    print(f"\nERROR: {type(e).__name__}: {e}")
    import traceback

    traceback.print_exc()
    print("\n--- Troubleshooting ---")
    print("1. Confirm Cell 5 loaded the dataset successfully.")
    print(f"2. Confirm the cloud endpoint is reachable: {OLLAMA_BASE_URL}")
    print("3. Inspect the chunk preview above for duplicated article content.")
    print("4. Compare the raw model output format if extraction still returns an empty graph.")

In [ ]:
# Cell 8: Execution guard
print("Only continue to the full benchmark run after Cell 7 succeeds on one article.")
print(f"Cached full-run outputs will be written under: {RESULTS_DIR}")

In [ ]:
# Cell 8: Full run — extract KGs for all articles (with resume support)

KG_CACHE_FILE = RESULTS_DIR / "extracted_kgs.json"

# Load cache if exists (resume)
if KG_CACHE_FILE.exists():
    extracted_kgs = json.loads(KG_CACHE_FILE.read_text(encoding="utf-8"))
    print(f"Resumed from cache: {len(extracted_kgs)} articles already done")
else:
    extracted_kgs = {}

for idx in tqdm(range(len(ds)), desc="Extracting KGs"):
    key = str(idx)
    if key in extracted_kgs:
        continue

    row = ds[idx]
    try:
        kg_dict = run_skgb_on_text(row["essay_content"])
        extracted_kgs[key] = {
            "topic": row["essay_topic"],
            "kg": kg_dict,
            "n_nodes": len(kg_dict["nodes"]),
            "n_edges": len(kg_dict["edges"]),
        }
    except Exception as e:
        print(f"\nERROR on article {idx} ({row['essay_topic']}): {e}")
        extracted_kgs[key] = {
            "topic": row["essay_topic"],
            "kg": {"nodes": [], "edges": []},
            "n_nodes": 0,
            "n_edges": 0,
            "error": str(e),
        }

    # Save after each article for resume
    KG_CACHE_FILE.write_text(
        json.dumps(extracted_kgs, indent=2, ensure_ascii=False), encoding="utf-8"
    )

print(f"\nDone! Extracted KGs for {len(extracted_kgs)} articles")
print(f"Saved to: {KG_CACHE_FILE}")

## MINE Evaluation

For each article:
1. Embed all KG node names using sentence-transformers
2. For each ground-truth fact, find top-k most similar nodes
3. Expand 2-hop neighborhood around those nodes
4. Ask LLM: "Can this fact be inferred from this subgraph context?" -> 0/1
5. Score = count(1) / 15

In [ ]:
# Cell 9: Evaluation functions

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")


def build_adjacency(kg_dict: dict) -> dict:
    """Build adjacency list from KG dict for neighborhood expansion."""
    adj = {}  # node_name -> [(relation, neighbor_name), ...]
    for edge in kg_dict.get("edges", []):
        src, tgt, rel = edge["source"], edge["target"], edge["relation"]
        adj.setdefault(src, []).append((rel, tgt))
        adj.setdefault(tgt, []).append((rel, src))  # undirected for retrieval
    return adj


def get_neighborhood(adj: dict, seed_nodes: list, hops: int = 2) -> list:
    """BFS expansion from seed nodes, return list of (node, relation, neighbor) triples."""
    visited = set(seed_nodes)
    frontier = set(seed_nodes)
    triples = []

    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            for rel, neighbor in adj.get(node, []):
                triples.append((node, rel, neighbor))
                if neighbor not in visited:
                    visited.add(neighbor)
                    next_frontier.add(neighbor)
        frontier = next_frontier

    return triples


def retrieve_context(fact: str, kg_dict: dict, top_k: int = 5, hops: int = 2) -> str:
    """Retrieve relevant KG context for a fact via embedding similarity + neighborhood."""
    nodes = kg_dict.get("nodes", [])
    if not nodes:
        return "(empty graph)"

    node_names = [n["name"] for n in nodes]
    node_embeddings = embed_model.encode(node_names)
    fact_embedding = embed_model.encode([fact])

    sims = cosine_similarity(fact_embedding, node_embeddings)[0]
    top_indices = np.argsort(sims)[-top_k:][::-1]
    seed_nodes = [node_names[i] for i in top_indices]

    adj = build_adjacency(kg_dict)
    triples = get_neighborhood(adj, seed_nodes, hops)

    if not triples:
        return f"Relevant nodes: {', '.join(seed_nodes)}\n(no edges found)"

    # Deduplicate and format
    seen = set()
    lines = []
    for s, r, o in triples:
        key = (s, r, o)
        if key not in seen:
            seen.add(key)
            lines.append(f"{s} --[{r}]--> {o}")

    return "\n".join(lines[:50])  # cap at 50 triples


def judge_fact(fact: str, context: str) -> int:
    """Ask LLM whether the fact can be inferred from the KG context. Returns 0 or 1."""
    prompt = (
        "You are evaluating a knowledge graph. Given the following subgraph context "
        "extracted from a knowledge graph, determine if the stated fact can be "
        "reasonably inferred from this context.\n\n"
        f"SUBGRAPH CONTEXT:\n{context}\n\n"
        f"FACT: {fact}\n\n"
        "Can this fact be inferred from the subgraph context above? "
        "Answer with ONLY '1' (yes, inferable) or '0' (no, not inferable)."
    )

    try:
        client = ollama.Client(host=OLLAMA_BASE_URL)
        response = client.chat(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature": 0.0,
                "num_ctx": NUM_CTX,          # FIX: explicit context window
            },
        )
        answer = response["message"]["content"].strip()
        # Parse: look for 1 or 0
        if "1" in answer[:5]:
            return 1
        return 0
    except Exception as e:
        print(f"  Judge error: {e}")
        return 0


def evaluate_article(kg_dict: dict, facts: list) -> dict:
    """Evaluate one article's KG against its ground-truth facts."""
    results = []
    for fact in facts:
        context = retrieve_context(fact, kg_dict, top_k=TOP_K, hops=HOP_DEPTH)
        score = judge_fact(fact, context)
        results.append({"fact": fact, "score": score, "context_snippet": context[:200]})

    total = len(facts)
    covered = sum(r["score"] for r in results)
    return {
        "total_facts": total,
        "covered_facts": covered,
        "mine_score": covered / total if total > 0 else 0.0,
        "details": results,
    }

print("Evaluation functions loaded.")


In [ ]:
# Cell 10: Run MINE evaluation on all articles

EVAL_CACHE_FILE = RESULTS_DIR / "mine_eval_results.json"

if EVAL_CACHE_FILE.exists():
    eval_results = json.loads(EVAL_CACHE_FILE.read_text(encoding="utf-8"))
    print(f"Resumed evaluation: {len(eval_results)} articles already evaluated")
else:
    eval_results = {}

for idx in tqdm(range(len(ds)), desc="Evaluating"):
    key = str(idx)
    if key in eval_results:
        continue
    if key not in extracted_kgs:
        continue

    row = ds[idx]
    kg_dict = extracted_kgs[key]["kg"]

    # Parse facts
    facts_raw = row["generated_queries"]
    if isinstance(facts_raw, str):
        facts = ast.literal_eval(facts_raw)
    else:
        facts = list(facts_raw)

    result = evaluate_article(kg_dict, facts)
    result["topic"] = row["essay_topic"]
    result["n_nodes"] = extracted_kgs[key]["n_nodes"]
    result["n_edges"] = extracted_kgs[key]["n_edges"]
    eval_results[key] = result

    # Save after each article
    EVAL_CACHE_FILE.write_text(
        json.dumps(eval_results, indent=2, ensure_ascii=False), encoding="utf-8"
    )

print(f"\nEvaluation complete! {len(eval_results)} articles evaluated.")

In [ ]:
# Cell 11: Summary statistics

scores = [eval_results[k]["mine_score"] for k in sorted(eval_results, key=int)]
topics = [eval_results[k]["topic"] for k in sorted(eval_results, key=int)]
n_nodes = [eval_results[k]["n_nodes"] for k in sorted(eval_results, key=int)]
n_edges = [eval_results[k]["n_edges"] for k in sorted(eval_results, key=int)]

df = pd.DataFrame({
    "topic": topics,
    "mine_score": scores,
    "n_nodes": n_nodes,
    "n_edges": n_edges,
})

print("=" * 60)
print("SKGB MINE BENCHMARK RESULTS")
print("=" * 60)
print(f"Articles evaluated: {len(scores)}")
print(f"Mean MINE score:    {np.mean(scores):.4f}")
print(f"Median:             {np.median(scores):.4f}")
print(f"Std:                {np.std(scores):.4f}")
print(f"Min:                {np.min(scores):.4f}")
print(f"Max:                {np.max(scores):.4f}")
print()

# Compare with baselines from the dataset
baseline_cols = ["kggen_accuracy", "graphrag_accuracy", "openie_accuracy"]
baselines = {}
for col in baseline_cols:
    if col in ds.column_names:
        vals = [float(ds[i][col]) for i in range(len(ds)) if ds[i][col] is not None]
        if vals:
            baselines[col.replace("_accuracy", "")] = np.mean(vals)

if baselines:
    print("Baseline comparison (mean MINE score):")
    print(f"  SKGB:     {np.mean(scores):.4f}")
    for name, val in baselines.items():
        print(f"  {name:10s}: {val:.4f}")

df.to_csv(RESULTS_DIR / "mine_summary.csv", index=False)
print(f"\nSaved to {RESULTS_DIR / 'mine_summary.csv'}")

In [ ]:
# Cell 12: Visualizations

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("SKGB MINE Benchmark Results", fontsize=14, fontweight="bold")

# 1. Score distribution histogram
ax = axes[0, 0]
ax.hist(scores, bins=15, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(np.mean(scores), color="red", linestyle="--", label=f"Mean={np.mean(scores):.3f}")
ax.set_xlabel("MINE Score")
ax.set_ylabel("Count")
ax.set_title("Score Distribution")
ax.legend()

# 2. Boxplot comparison with baselines
ax = axes[0, 1]
box_data = {"SKGB": scores}
for col in baseline_cols:
    if col in ds.column_names:
        name = col.replace("_accuracy", "")
        vals = [float(ds[i][col]) for i in range(len(ds)) if ds[i][col] is not None]
        if vals:
            box_data[name] = vals
bp = ax.boxplot(box_data.values(), labels=box_data.keys(), patch_artist=True)
colors = ["steelblue", "#e8a838", "#56b356", "#d45656"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("MINE Score")
ax.set_title("SKGB vs Baselines")

# 3. Score vs graph size scatter
ax = axes[1, 0]
ax.scatter(n_edges, scores, alpha=0.6, c="steelblue", edgecolors="white", s=40)
ax.set_xlabel("Number of Edges")
ax.set_ylabel("MINE Score")
ax.set_title("Score vs Graph Size")

# 4. Per-article bar chart (sorted)
ax = axes[1, 1]
sorted_idx = np.argsort(scores)
ax.barh(range(len(scores)), [scores[i] for i in sorted_idx], color="steelblue", alpha=0.7)
ax.set_xlabel("MINE Score")
ax.set_ylabel("Article (sorted)")
ax.set_title("Per-Article Scores")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "mine_results_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {RESULTS_DIR / 'mine_results_overview.png'}")

In [ ]:
# Cell 13: Qualitative analysis — best & worst articles

sorted_articles = sorted(eval_results.items(), key=lambda x: x[1]["mine_score"])

print("=" * 60)
print("TOP 3 (best scoring)")
print("=" * 60)
for key, res in sorted_articles[-3:][::-1]:
    print(f"\n[Article {key}] {res['topic']}")
    print(f"  Score: {res['mine_score']:.3f} ({res['covered_facts']}/{res['total_facts']})")
    print(f"  Graph: {res['n_nodes']} nodes, {res['n_edges']} edges")
    covered = [d["fact"] for d in res["details"] if d["score"] == 1]
    print(f"  Covered facts: {covered[:3]}")

print()
print("=" * 60)
print("BOTTOM 3 (worst scoring)")
print("=" * 60)
for key, res in sorted_articles[:3]:
    print(f"\n[Article {key}] {res['topic']}")
    print(f"  Score: {res['mine_score']:.3f} ({res['covered_facts']}/{res['total_facts']})")
    print(f"  Graph: {res['n_nodes']} nodes, {res['n_edges']} edges")
    missed = [d["fact"] for d in res["details"] if d["score"] == 0]
    print(f"  Missed facts: {missed[:3]}")

In [ ]:
# Cell 14: Export final results

# Compact summary JSON
summary = {
    "benchmark": "MINE",
    "pipeline": "SKGB (chunking + itext2kg ATOM)",
    "llm_model": LLM_MODEL,
    "embeddings_model": EMBEDDINGS_MODEL,
    "config": {
        "ent_threshold": ENT_THRESHOLD,
        "rel_threshold": REL_THRESHOLD,
        "min_chunk_words": MIN_CHUNK_WORDS,
        "max_chunk_words": MAX_CHUNK_WORDS,
        "overlap_words": OVERLAP_WORDS,
        "top_k": TOP_K,
        "hop_depth": HOP_DEPTH,
    },
    "results": {
        "n_articles": len(scores),
        "mean_mine_score": float(np.mean(scores)),
        "median_mine_score": float(np.median(scores)),
        "std_mine_score": float(np.std(scores)),
        "min_mine_score": float(np.min(scores)),
        "max_mine_score": float(np.max(scores)),
    },
    "baselines": baselines,
    "timestamp": datetime.datetime.now().isoformat(),
}

(RESULTS_DIR / "benchmark_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

print("Exported:")
print(f"  {RESULTS_DIR / 'extracted_kgs.json'}")
print(f"  {RESULTS_DIR / 'mine_eval_results.json'}")
print(f"  {RESULTS_DIR / 'mine_summary.csv'}")
print(f"  {RESULTS_DIR / 'benchmark_summary.json'}")
print(f"  {RESULTS_DIR / 'mine_results_overview.png'}")
print("\nDone!")